# 03 The market characteristics: eight measures from prices, returns and trading

**What this notebook does.** For every member-month of the panel's window it measures eight characteristics from CRSP's monthly and daily stock files and from Kenneth French's daily market return: size, momentum, reversal, dollar volume, volatility, the largest daily return, the stock's sensitivity to the market's moves, called beta, and the volatility of its return that the market does not explain, called idiosyncratic volatility. Each is measured at the end of month t-1, so that it uses only what was known when month t began. It counts by year how many member-months have each one.

**Why it comes here.** The models of milestones 2 and 3 forecast next month's return from twelve characteristics. Eight of them come from prices and trading, and the accounting four come in notebook 04. Gate G1 (b), judged in notebook 05, asks each to be present for at least 90 per cent of each month's members, and this notebook shows where each stands by year.

**What it judges.** No gate and no expectation with a level. Notebook 05 judges coverage on these columns and runs one more test on them, the look-ahead test: a check that no characteristic used a number dated after the day it is measured on.

**The data.** CRSP, from Wharton Research Data Services, accessed through Queen Mary University of London, extracted on 28 September 2026, and Kenneth French's data library, downloaded at run time.

## Terms used in this notebook

| Term | Meaning |
|---|---|
| characteristic | a number measured for one stock in one month, used to forecast its return in the next month |
| month t, month t-1 | month t is the month whose return is forecast. Month t-1 is the month before. Every characteristic is measured on the last trading day of month t-1 |
| trading day | a day on which the daily stock file has rows |
| size | the log of the start-of-month capitalisation: the absolute price times shares outstanding at the end of month t-1, in thousands of dollars (CRSP's guide, pp. 70 and 79) |
| momentum | the stock's return compounded over the months t-12 to t-2. Month t-1 is left out. It is missing when the price at t-13 or the return at t-2 is missing |
| reversal | the stock's return in month t-1 |
| share volume | the number of shares traded in a month. CRSP's monthly file records it in hundreds of shares (CRSP's guide, p. 91) |
| dollar volume | the log of the price at the end of month t-1 times the month's share volume in shares |
| daily return | the stock's return on one trading day, with dividends |
| volatility | the standard deviation of the daily returns in month t-1, with at least 15 of them |
| largest daily return | the highest daily return in month t-1, with at least 15 daily returns |
| risk-free rate | French's daily return on one-month Treasury bills |
| excess return | a return less the risk-free rate of the same day |
| market excess return | French's daily return on the market of all CRSP stocks, each weighted by its market value, less the risk-free rate |
| regression | a fit of a straight line, y = a + b x, by least squares: the a and b that make the sum of squared distances from the points to the line smallest |
| beta | the slope b of the stock's daily excess return on the market excess return, over the 252 trading days to the end of month t-1, with at least 120 days on which both are present |
| residual | the distance from one point to the fitted line, y - a - b x |
| idiosyncratic volatility | the standard deviation of the same regression's residuals: the square root of their sum of squares over n - 2, where n is the number of days, because the fit estimates two numbers |
| coverage | the share of the member-months in which a characteristic is present |
| verdict | met or not met: what a gate's level, or an expectation's, says of a result. An expectation with no verdict is set beside its result and judged by nothing |
| provenance record | where a result's inputs came from: for French's files, each file's address, the time it was read and its SHA-256 |

## What I expect to see, and where it comes from

1. Size is present in all but 10 of the panel's 204,520 member-months: Stock-Level Tracking's `delistings_by_year.csv` counts 10 member-months without a start-of-month capitalisation from 1991 to 2024.
2. Volatility, the largest daily return, beta and idiosyncratic volatility are present for more than 98 per cent of member-months in every year. In Stock-Level Tracking's `daily_coverage_by_year.csv` at least 99.64 per cent of member-days have a daily return in every year from 1990, the lowest in 2000. A member needs 15 returns in a month and 120 in 252 days, and a stock that entered the index from a recent listing can fall short of 120.
3. Momentum needs 13 months of history, so it is missing for a stock listed less than 13 months before. The S&P 500 adds few such stocks, so momentum is expected above 97 per cent in every year, a figure not read from the data.
4. The mean of beta across each year's member-months lies between 0.7 and 1.3 in every year. The members make up most of the market's value, so their betas weighted by value average close to one. The mean here gives each member the same weight, so it can stray further, most likely around 2000, when the largest technology stocks moved the market and many other members did not move with them. With no verdict.
5. The median volatility across members is highest in 2008 or 2009. The volatility of a member-month in year y is measured in the month before, so the falls of late 2008 and early 2009 reach both years. With no verdict.

Gate G1 (b) on these columns is judged in notebook 05. Running time: about five minutes, most of it spent reading the daily stock file a year at a time.

In [ ]:
import hashlib
import json
import pathlib
import sys
import time

# The package lives in src/ beside this folder. The notebook finds the repository's root from where it runs.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "mlrp").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from mlrp import constants as C
from mlrp import data as D
from mlrp.source import extraction as X
from mlrp.source import layer as L
from mlrp.source import returns as R
from mlrp.source import universe as U

# Stops with one sentence when the data root is not set, when the manifest is not the one the constants fix,
# or when one of the study's eight tables is not the one they record.
root, manifest = D.open_extraction()
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}", "| entries of the eight tables", C.MANIFEST_ENTRIES_SHA256[:12])

In [ ]:
g1a = D.read_result("g1a.json")
if not g1a["g1a_met"]:
    raise SystemExit("Gate G1 (a) is not met, so nothing after notebook 02 runs.")
print("Gate G1 (a): met, in notebook 02")

## 1. The member-months of the panel's window

The universe is built again, as in notebook 02, and cut to the panel's window, January 1991 to December 2024. Each member-month carries `date_prev`, the last trading day of month t-1, the day every characteristic is measured on.

In [ ]:
universe = L.load_universe(root)
ulong = universe["ulong"]
inside = pd.PeriodIndex(ulong["month"], freq="M")
members = ulong[(inside >= pd.Period(C.PANEL_FIRST_MONTH, "M")) & (inside <= pd.Period(C.PANEL_LAST_MONTH, "M"))].reset_index(drop=True)
permnos = np.sort(members["permno"].unique())
print(f"member-months {len(members):,} | months {members['month'].nunique()} | stocks {len(permnos):,}")

## 2. The four characteristics from the monthly file

The monthly file is read for the panel's stocks over every month it holds, since momentum in January 1991 reaches back to December 1989. `characteristics.monthly_characteristics` gives size, momentum, reversal and dollar volume for every member-month. Momentum is the copied function of Stock-Level Tracking, which follows Kenneth French's definition.

In [ ]:
t0 = time.time()
monthly, bad_monthly = D.read_table(root, "crsp_msf")
monthly = monthly[monthly["permno"].isin(permnos)].reset_index(drop=True)
delistings, bad_delistings = D.read_table(root, "crsp_msedelist")
from mlrp import characteristics as K
monthly_chars = K.monthly_characteristics(members, monthly, delistings)
print(f"monthly rows {len(monthly):,} | {time.time() - t0:.0f} s")
print("date values present that did not parse:", {**bad_monthly, **bad_delistings} or "none")
print(monthly_chars[list(K.MONTHLY)].notna().mean().round(4).to_string())

## 3. French's daily market return and risk-free rate

French's daily data hold the market excess return and the risk-free rate, in per cent, which the copied loader turns into decimals. The files are downloaded once into `data/french/`, which git ignores, and their record, with each file's SHA-256 and the date in its header, goes into `provenance_french.json`.

In [ ]:
factors, risk_free, french_records = D.french_daily()
market_excess = factors["Mkt-RF"]
print("French's daily series from", market_excess.index.min().date(), "to", market_excess.index.max().date(), "|", len(market_excess), "days")
for r in french_records:
    print(r["filename"], "| SHA-256", r["zip_sha256"][:16], "|", r["vintage_line"])

## 4. The daily returns

The daily stock file is read a year at a time, from 1990, for the panel's stocks, and held in memory as one table of trading days by stock. It is never written. The calendar of trading days is the file's own, and French's series are aligned to it. A trading day with no market return in French's file would leave that day out of every regression, and those days are counted.

In [ ]:
t0 = time.time()
years = list(range(int(C.DAILY_FILE_START[:4]), int(C.PANEL_LAST_MONTH[:4]) + 1))
daily, calendar = L.daily_matrix(root, permnos, years, progress=True)
print(f"trading days {len(calendar):,}, {calendar[0].date()} to {calendar[-1].date()} | stocks {daily.shape[1]:,} | {time.time() - t0:.0f} s")
print("trading days without French's market return:", int(market_excess.reindex(calendar).isna().sum()))

## 5. The four characteristics from the daily file

`characteristics.daily_characteristics` measures volatility and the largest daily return on the trading days of month t-1, and beta and idiosyncratic volatility on the 252 trading days to its last day. Each month is computed from its own days alone.

**A worked example of the regression.** Take three days on which the market's excess return was 1, 0 and -1 per cent and a stock's was 2, 0.5 and -1 per cent. The line that fits best has slope b = 1.5 and intercept a = 0.5 per cent: the fitted values 2, 0.5 and -1 equal the stock's, so the residuals are 0, 0 and 0, and the idiosyncratic volatility is 0. A stock with the same slope and residuals of +1, -2 and +1 per cent has a sum of squares of 6, so its idiosyncratic volatility is the square root of 6 / (3 - 2), 2.45 per cent a day. The study fits the same line over up to 252 days.

In [ ]:
t0 = time.time()
daily_chars = K.daily_characteristics(members, daily, market_excess, risk_free)
print(f"{time.time() - t0:.0f} s")
print(daily_chars[list(K.DAILY)].notna().mean().round(4).to_string())

## 6. Coverage and medians by year

Two tables by year, both aggregates over each year's member-months: the share with each characteristic present, and the median of each across members. The mean of beta is printed beside them for expectation 4.

In [ ]:
market = members[["month", "permno"]].merge(monthly_chars, on=["month", "permno"]).merge(daily_chars, on=["month", "permno"])
names = list(K.MONTHLY) + list(K.DAILY)
year = pd.PeriodIndex(market["month"], freq="M").year
coverage = market[names].notna().groupby(year).mean().round(6)
coverage.insert(0, "member_months", market.groupby(year).size())
coverage.index.name = "year"
medians = market[names].groupby(year).median().round(6)
medians.index.name = "year"
beta_mean = market["beta"].groupby(year).mean().round(3)
pd.set_option("display.width", 200)
print(coverage.to_string())
print()
print(medians.to_string())
print()
print("mean beta by year, lowest", beta_mean.min(), "in", int(beta_mean.idxmin()), "| highest", beta_mean.max(), "in", int(beta_mean.idxmax()))
print("the three years of the highest median volatility:", [int(y) for y in medians["volatility"].sort_values(ascending=False).index[:3]])

## 7. What is kept and what is committed

The eight characteristics of every member-month are kept in `local/market_characteristics.parquet` for notebook 05. The folder is ignored by git and stays on the machine that holds the extraction. Three files go into `results/20260928/`: the coverage by year, the medians by year and a summary. French's provenance record goes beside them.

In [ ]:
D.write_local("market_characteristics.parquet", market.assign(month=market["month"].astype(str)))
written = [D.write_results("market_coverage_by_year.csv", frame=coverage.reset_index()),
           D.write_results("market_medians_by_year.csv", frame=medians.reset_index()),
           D.write_results("market_summary.json", record={
               "written_by": "notebooks/03_market_characteristics.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
               "member_months": int(len(market)), "stocks": int(len(permnos)), "monthly_rows": int(len(monthly)),
               "trading_days": int(len(calendar)), "first_trading_day": str(calendar[0].date()), "last_trading_day": str(calendar[-1].date()),
               "trading_days_without_market_return": int(market_excess.reindex(calendar).isna().sum()),
               "present": {c: int(market[c].notna().sum()) for c in names},
               "mean_beta_by_year": {int(k): float(v) for k, v in beta_mean.items()},
               "date_values_not_parsed": {**bad_monthly, **bad_delistings}}),
           D.write_french_provenance(french_records)]
for path in written:
    print("written:", path.relative_to(ROOT))
results = pd.DataFrame([
    ("1. size missing in member-months", "10", str(int(market["size"].isna().sum()))),
    ("2. lowest yearly coverage of the four daily ones", "above 0.98", f"{coverage[list(K.DAILY)].min().min():.4f}"),
    ("3. lowest yearly coverage of momentum", "above 0.97", f"{coverage['momentum'].min():.4f}"),
    ("4. mean beta by year", "0.7 to 1.3", f"{beta_mean.min():.3f} to {beta_mean.max():.3f}"),
    ("5. year of the highest median volatility", "2008 or 2009", str(int(medians["volatility"].idxmax()))),
], columns=["expectation", "expected", "result"])
print(results.to_string(index=False))

## What was learned, and what could be wrong

- The table above sets each expectation beside its result. None has a verdict here: coverage is judged by gate G1 (b) in notebook 05.
- Beta and idiosyncratic volatility are measured against French's market, the return of all CRSP stocks weighted by their market value, and over a year of daily returns. Ang, Hodrick, Xing and Zhang (2006) measure idiosyncratic volatility within one month, against the market and two portfolios that stand for size and value. The two measures rank stocks in a similar order, and this study's version is the one in the constants.
- Daily returns in the 1990s include stocks traded on NASDAQ, where volume was counted differently before 2004. Volume enters only dollar volume, and dollar volume is ranked within each month, so a difference that moves every NASDAQ stock alike moves their ranks against NYSE stocks. The panel does not correct for it.
- Each characteristic is measured from days up to the last trading day of month t-1. Notebook 05 builds 24 months a second time, from inputs cut at that day, and compares the two builds, which is the test that nothing later was used.